# 05 — Graded relevance labeling

Builds the learning-to-rank training target from real realized forward returns. Each quarter's entire cross-section is rank-ordered and bucketed into 10 rank-ordered relevance grades (0 = worst decile, 9 = best). This replaced a binary "top-10 winners" label outright.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## Why graded, not binary

The previous target was binary: a stock was one of the quarter's 10 best, or indistinguishable from everything else. That could not tell the 400th stock from the 1,400th — useless for a system whose entire output is a full 1..N ranking. The graded target orders the whole cross-section, which is what LambdaRank needs.

Grades are assigned by **rank position**, not return magnitude:

```
grade = n_relevance_grades - 1 - floor((rank - 1) * n_relevance_grades / valid_count)
```

so the target stays comparable across quarters — a 5% return may be top-decile in one quarter and median in another, and LambdaRank only compares items inside their own quarter (query group) anyway.

## Real forward-return outcomes for one quarter

Entry is the last close on or before the cycle's feature timestamp; exit is the last close on or before the next cycle's start. Neither may use a price dated after the exit cutoff. A missing or non-positive entry price yields `None` — explicitly "not computable", never NaN and never 0.0.

In [ ]:
if AVAILABLE:
    from datetime import datetime, time

    from atlas_quant.strategies.multi_factor_ranking_ml.forward_return import (
        build_forward_return_outcome,
    )

    bundle = rd.slice_bundle()
    cycle = bundle.cycles[-2]
    next_cycle = bundle.cycles[-1]
    exit_cutoff = datetime.combine(next_cycle.quarter_start, time.min)
    observations = bundle.feature_results[cycle.quarter_start].observations

    outcomes = [
        build_forward_return_outcome(
            obs.instrument_id,
            cycle.quarter_start,
            obs.feature_timestamp,
            next_cycle.quarter_start,
            bundle.prices_by_instrument.get(obs.instrument_id, ()),
            exit_cutoff,
        )
        for obs in observations
    ]
    computable = [o for o in outcomes if o.clipped_return is not None]
    print(f"quarter {cycle.quarter_start} -> {next_cycle.quarter_start}")
    print(f"  observations           {len(outcomes):,}")
    print(f"  computable forward ret {len(computable):,}")
    print(f"  not computable         {len(outcomes) - len(computable):,}")
    print()
    example = computable[0]
    print("example outcome:")
    print("  instrument   ", example.instrument_id.symbol)
    print("  entry / exit ", example.entry_price, "/", example.exit_price)
    print("  raw / clipped", round(example.raw_return, 6), "/", round(example.clipped_return, 6))
    print("  knowable at  ", example.label_available_at)
else:
    bundle = None
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Return capping

Returns are clipped at ±`return_cap` before ranking, so a single extreme move cannot dominate the ordering.

In [ ]:
if AVAILABLE:
    cap = rd.config().return_cap
    clipped_high = sum(1 for o in computable if o.raw_return > cap)
    clipped_low = sum(1 for o in computable if o.raw_return < -cap)
    print(f"return_cap = ±{cap}")
    print(f"  clipped at the top   : {clipped_high:,}")
    print(f"  clipped at the bottom: {clipped_low:,}")
    raws = sorted(o.raw_return for o in computable)
    print(f"  raw return range     : {raws[0]:.3f} .. {raws[-1]:.3f}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Assigning graded relevance

In [ ]:
if AVAILABLE:
    from collections import Counter

    from atlas_quant.strategies.multi_factor_ranking_ml.labeling import assign_quarterly_relevance

    config = rd.config()
    graded = assign_quarterly_relevance(
        outcomes, cycle.quarter_start, n_relevance_grades=config.n_relevance_grades
    )
    print(f"quarter_end        {graded.quarter_end}")
    print(f"n_relevance_grades {graded.n_relevance_grades}")
    print(f"valid              {graded.valid_count:,}")
    print(f"excluded           {graded.excluded_count:,}  (no computable return -> grade 0)")
    print()
    print(f"{'grade':>6} {'count':>7}   (9 = best-performing decile)")
    for grade, count in sorted(Counter(a.relevance for a in graded.assignments).items(), reverse=True):
        bar = "#" * round(count / max(1, graded.valid_count) * 200)
        print(f"{grade:>6} {count:>7,}   {bar}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## This is a graded target, not a binary one

Asserted directly: the real quarter produces every grade in `0..n_relevance_grades-1`, and the grades are monotone in realized forward return.

In [ ]:
if AVAILABLE:
    import statistics

    grades_present = {a.relevance for a in graded.assignments}
    print("distinct grades present:", sorted(grades_present))
    assert grades_present == set(range(config.n_relevance_grades))

    by_grade = {}
    for assignment in graded.assignments:
        if assignment.clipped_return is not None:
            by_grade.setdefault(assignment.relevance, []).append(assignment.clipped_return)

    print()
    print(f"{'grade':>6} {'n':>6} {'mean clipped return':>21}")
    means = []
    for grade in sorted(by_grade, reverse=True):
        mean = statistics.mean(by_grade[grade])
        means.append(mean)
        print(f"{grade:>6} {len(by_grade[grade]):>6,} {mean:>21.4f}")
    assert means == sorted(means, reverse=True), "grades must be monotone in forward return"
    print()
    print("OK: mean realized forward return decreases monotonically from grade 9 to 0.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## The labeled quarters the model actually trains on

In [ ]:
if AVAILABLE:
    from collections import Counter

    print(f"{'quarter':<12} {'labeled rows':>13} {'distinct grades':>16}")
    for quarter_start in sorted(bundle.labeled_quarters):
        rows = bundle.labeled_quarters[quarter_start]
        print(f"{str(quarter_start):<12} {len(rows):>13,} "
              f"{len({r.relevance for r in rows}):>16}")
    print()
    print("The final cycle has no labeled quarter: its forward return is not yet realized.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- On real data every quarter in the slice populates all 10 relevance grades, and mean realized forward return is monotone decreasing from grade 9 to grade 0 — the target orders the cross-section as intended.
- Instruments with no computable forward return are excluded from ranking and receive grade 0 explicitly, rather than being dropped silently or given a fabricated return.
- Return capping bites on a real minority of names each quarter, which is the point: one takeover or one collapse should not define a quarter's ordering.

## Limitations

- Grade 0 conflates "genuinely worst decile" with "no computable forward return". Both are legitimately uninformative for a long ranking, but they are not the same fact, and the model cannot tell them apart.
- Forward returns are price-only: no dividends beyond whatever the `split_dividend_adjusted` series already embeds, and no delisting handling. A delisted name simply stops having prices.
- Labels are computed only over the bounded slice, using the price window cached for it.